**1. The History & Evolution of Missing Data in Python & Data Science**

- Systems have been designed to store tabular data

*How do you represent a value that isn't there?*

*Two Primary Strategies*

- The masking approach: Allocating a boolean array alongside the dataset to track cells that a valid or missing

- The sentinel approach: A special placeholder within the data itself (-9999)

*What was the Python approach to missingness?*

- In came Pandas: Pandas adopted a sentinel based approach to missing values.

*None: Python's native null object*

*NaN(Not a Number): A special floating point value*

- NaN forced integer columns with missing values to automatically convert into floating point numbers

- To solve for this, Pandas introduced a nulable data type(Int32 or boolean)

**2. A Step-by-Step Walkthrough: How Missing Data is Represented in Python & Pandas**

In [2]:
# Packages of interest
import pandas as pd
import numpy as np

In [3]:
# Python's Native Sentinel - None
# Create a Series with None
series_none = pd.Series([1, None, 3, 4])
series_none

0    1.0
1    NaN
2    3.0
3    4.0
dtype: float64

In [4]:
# NumPy's Numerical Sentinel - NaN
# Create a Series with NaN
series_nan = pd.Series([1, np.nan, 3, 4])
series_nan

0    1.0
1    NaN
2    3.0
3    4.0
dtype: float64

In [5]:
# The Virus Effect - NaN
vals = np.array([1, np.nan, 3, 4])

In [6]:
vals.sum(), vals.min(), vals.max()

(np.float64(nan), np.float64(nan), np.float64(nan))

In [7]:
vals

array([ 1., nan,  3.,  4.])

In [8]:
np.nansum(vals), np.nanmin(vals), np.nanmax(vals)

(np.float64(8.0), np.float64(1.0), np.float64(4.0))

In [9]:
# Modern Pandas Nullable Data Type
nullable_series = pd.Series([1, np.nan, 2, None, pd.NA], dtype='Int32')
nullable_series

0       1
1    <NA>
2       2
3    <NA>
4    <NA>
dtype: Int32

In [10]:
# Core Pandas Methods for Handling Data Types
# Pandas has four primary methods for detecting, removing, and replacing missing data
# 1. Detecting Nulls - isnull()/notnull(): This returns a boolean mask indicating where values are missing
# 2. Dropping Nulls - dropna(): Filters out missing values
# 3. Filling Nulls - fillna(): Replaces null entries with a specified value or method

**Why is missing data important in the ML Workflow?**

- Human Error
- Broken sensors
- Internet connection outages
- Evasiveness

**The Mental Models for Dealing with missing data**
- Is this a jigsaw puzzle scenario?
- Is this a recipe scenario?

**Why does missing data affect ML workflows?**
- Computer programs (ML Algorithms) expect numbers not blank spaces
- Missing data introduces bias: Removing rows containing missing values will leave us with data that incorrectly suggests a different situation.
- Loss of Information: Missing values indicates loss of information. While dealing with missing values, getting rid of rows with missing information destroys a portion of the data.

**Why is the data missing?**
- Missing by Accident: Random error (MCAR).
- Missing by nature or logic: The question in the survey may not apply to them (MAR).
- Missing on purpose: Sensitive information or Privacy (MNAR).

In [12]:
# Hands-on Walkthrough
unemp_data = pd.read_csv('nigeria_unemployment_missing_data.csv')
unemp_data.head()

In [13]:
# Auditing missing values across all columns
print("--- Missing Values Count Per Column ---")
print(unemp_data.isnull().sum())

--- Missing Values Count Per Column ---
Age                       0
Gender                    0
Region                  250
Location                  0
Education_Level        1127
Employment_Status         0
Years_Of_Experience     500
Monthly_Income_NGN      408
dtype: int64


In [15]:
# Pruning (Dropping Missing Data)
# If a Region or Location is blank, guessing the state or zone creates false geographical data
# We prune rows where essential location tags are missing
clean_location_data = unemp_data.dropna(subset=['Region', 'Location'])

In [16]:
print(f"Rows before dropping missing location: {len(unemp_data)}")
print(f"Rows after dropping missing location: {len(clean_location_data)}")

Rows before dropping missing location: 5000
Rows after dropping missing location: 4750


In [17]:
# Numerical Patching (Mean vs. Median Imputation)
# Impute missing values with a central tendency metric
# Mean (Average): Best used if the data is evenly distributed without wild extremes
# Median (Middle Value): Best used when data contains extreme outliers

# Inspect our income distribution for outliers/distortions
income_median = unemp_data['Monthly_Income_NGN'].median()
income_mean = unemp_data['Monthly_Income_NGN'].mean()
print(f"Median Income: {income_median:.2f}")
print(f"Mean Income: {income_mean:.2f}")

Median Income: 42908.88
Mean Income: 58114.54


In [18]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(unemp_data['Monthly_Income_NGN'].dropna(), kde=True)
plt.title('Distribution of Monthly Income (NGN)')
plt.xlabel('Monthly Income (NGN)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [19]:
# Categorical Patching (Mode vs. 'Unknown' / 'Unspecified')
# Strategy for dealing with missing education level data: Flag as unassigned or unspecified
unemp_data['Education_Level'] = unemp_data['Education_Level'].fillna('Unspecified')

In [20]:
unemp_data['Education_Level'].unique()

array(['Unspecified', 'Primary', 'Secondary', 'Tertiary'], dtype=object)

In [21]:
# The Group-Based Imputation
# Calculate the average of the median within a specific subgroup
# Then, fill in accordingly
# Calculate the average experience per employment status group
group_experience = unemp_data.groupby('Employment_Status')['Years_Of_Experience'].mean().round(2)

In [22]:
group_experience

Employment_Status
Employed         21.72
Underemployed    22.38
Unemployed       20.83
Name: Years_Of_Experience, dtype: float64

In [23]:
# Then, fill in accordingly
unemp_data['Years_Of_Experience'] = unemp_data['Years_Of_Experience'].fillna(group_experience)